#  Scalable Visualization and Explainability of Synthetic Datasets

In [ ]:
%load_ext jupyter_black

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objs as go
import statsmodels.api as sm
import seaborn as sns
import time
import warnings
import logging

from matplotlib.ticker import ScalarFormatter
from scipy.stats import ks_2samp, mannwhitneyu, ttest_ind
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.manifold import (
    TSNE,
    LocallyLinearEmbedding,
    Isomap,
    MDS,
    SpectralEmbedding,
)
from sklearn.linear_model import LinearRegression
from umap import UMAP
from tsnecuda import TSNE as TSNE_GPU

# from cuml.manifold.umap import UMAP
# from cuml.manifold import TSNE

from vizdataquality import (
    calculate as vdqc,
    datasets as vdqd,
    plot as vdqp,
    report as vdqr,
)

In [ ]:
from plotly.offline import init_notebook_mode

init_notebook_mode(connected=True)

In [ ]:
pip install -q plotly

In [ ]:
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

In [ ]:
# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)

In [ ]:
%matplotlib inline

warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
sns.set(
    context="notebook",
    rc={"figure.figsize": (12, 10)},
    palette=sns.color_palette("tab10", 10),
)

In [ ]:
!python --version

In [ ]:
start = time.time()

## Data Understanding

### Real Data - Insurance

In [ ]:
insurance_rdf = pd.read_csv("../data/raw/data/insurance/real/insurance.csv")

In [ ]:
insurance_rdf.head()

### Synthetic Data  - Insurance

In [ ]:
logging.info("Starting to read the insurance dataset.")

start_time = time.time()
try:
    insurance_sdf = pd.read_csv("../data/raw/data/insurance/synth/insurance_1M.csv", index_col=0)
    elapsed = time.time() - start_time
    logging.info(
        f"Successfully read insurance dataset. Shape: {insurance_sdf.shape}. Time taken: {elapsed:.2f} seconds."
    )
except Exception as e:
    logging.error("Failed to read the insurance dataset.", exc_info=True)

In [ ]:
insurance_sdf = pd.read_csv("../data/raw/data/insurance/synth/insurance_1M.csv", index_col=0)

In [ ]:
insurance_sdf.head()

In [ ]:
insurance_sdf1 = pd.read_csv(
    "../data/raw/data/insurance/synth/insurance_100K.csv",
    index_col=0,
)

In [ ]:
insurance_sdf1.head()

### Descriptive Statistics

#### Real Data

In [ ]:
insurance_rdf.describe()

#### Synthethic Data

In [ ]:
# 1M
insurance_sdf.describe()

In [ ]:
# 100K
insurance_sdf1.describe()

## Data Validation & Profiling

In [ ]:
# data validation real data
vdqc.calc(insurance_rdf)

In [ ]:
## Data validation synthetic data
vdqc.calc(insurance_sdf)

In [ ]:
# data validation synthetic data
vdqc.calc(insurance_sdf1)

## EDA

### Violin Plot Real  Data

In [ ]:
def violin_plot(data, **kwargs):

    numeric_data = data.select_dtypes("float")
    fig, ax = plt.subplots(1, len(numeric_data.columns), **kwargs)

    for i, col in enumerate(numeric_data.columns):
        ax[i].violinplot(numeric_data[col], showmedians=True, orientation="vertical")
        ax[i].set_title(f"Distribution of {col}")
        ax[i].set_ylabel("Value")
        ax[i].set_xticks([1])
        ax[i].set_xticklabels([col])

    plt.tight_layout()
    plt.show()

In [ ]:
violin_plot(insurance_rdf[["BMI", "CHARGES"]], figsize=(6, 3))

### Violin Plot Synthetic  Data - 1M rows

In [ ]:
violin_plot(insurance_sdf[["BMI", "CHARGES"]], figsize=(6, 3))

### Violin Plot 100k

In [ ]:
violin_plot(insurance_sdf1[["BMI", "CHARGES"]], figsize=(6, 3))

### Histogram plot

In [ ]:
def histogram_plot(data, num_rows, num_columns):

    numeric_data = data.select_dtypes("float")
    fig, ax = plt.subplots(num_rows, num_columns, figsize=(5 * num_columns, 3 * num_rows))
    ax = ax.flatten()

    for i, col in enumerate(numeric_data.columns):
        sns.histplot(numeric_data[col], ax=ax[i], kde=True)
        ax[i].set_title(f"Distribution of {col}")
        ax[i].set_ylabel("Value")

    plt.tight_layout()
    plt.show()

#### Real Data

In [ ]:
histogram_plot(data=insurance_rdf, num_rows=1, num_columns=2)

#### Synthethic Data 1M

In [ ]:
histogram_plot(data=insurance_sdf[["BMI", "CHARGES"]], num_rows=1, num_columns=2)

#### Synthethic Data 100k

In [ ]:
histogram_plot(data=insurance_sdf1[["BMI", "CHARGES"]], num_rows=1, num_columns=2)

### Categorical Plots

In [ ]:
def cat_plot(df, **kwargs):

    cat = df.select_dtypes("object")
    fig, ax = plt.subplots(1, len(cat.columns), **kwargs)

    for i, col in enumerate(cat):
        count = cat[col].value_counts().reset_index()
        count.columns = [col, "count"]
        sns.barplot(x=col, y="count", data=count, ax=ax[i])
        ax[i].set_title(f"Distribution of {col}")
        ax[i].set_ylabel("Value")
        ax[i].tick_params(axis="x", rotation=45)

    plt.tight_layout()
    plt.show()

####  Real Data

In [ ]:
cat_plot(insurance_rdf, figsize=(8, 3))

#### Synthethic Data - 1M

In [ ]:
cat_plot(insurance_sdf, figsize=(8, 3))

#### Synthethic Data - 100K

In [ ]:
cat_plot(insurance_sdf1, figsize=(8, 3))

### Comparison of Distributions between Real and Synthethic Data

In [ ]:
def plot_distribution_comparison(
    df1, df2, column="VALUE", label1="Dataset 1", label2="Dataset 2", **kwargs
):
    figsize = kwargs.get("figsize", (8, 5))
    fig, ax = plt.subplots(1, 1, figsize=figsize)

    # Histogram comparison
    sns.histplot(
        df1[column],
        bins=30,
        color="blue",
        label=label1,
        stat="density",
        alpha=0.5,
        element="step",
        ax=ax,
    )
    sns.histplot(
        df2[column],
        bins=30,
        color="red",
        label=label2,
        stat="density",
        alpha=0.5,
        element="step",
        ax=ax,
    )
    ax.set_xlabel(column)
    ax.set_ylabel("Density")
    ax.set_title(f"Histogram Comparison of {column}")
    ax.legend()

    # --- Statistical Tests ---
    data1 = df1[column].dropna()
    data2 = df2[column].dropna()

    # Kolmogorov-Smirnov Test
    ks_stat, ks_p = ks_2samp(data1, data2)

    # Mann-Whitney U Test
    mw_stat, mw_p = mannwhitneyu(data1, data2, alternative="two-sided")

    # T-test
    t_stat, t_p = ttest_ind(data1, data2, equal_var=False)

    # Annotate plot with p-values
    textstr = "\n".join((f"KS p = {ks_p:.3g}", f"MW U p = {mw_p:.3g}", f"T-test p = {t_p:.3g}"))

    ax.text(
        0.98,
        0.95,
        textstr,
        transform=ax.transAxes,
        fontsize=10,
        verticalalignment="top",
        horizontalalignment="right",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.6),
    )

    plt.tight_layout()
    plt.show()

In [ ]:
for col in insurance_rdf.select_dtypes("float").columns:
    plot_distribution_comparison(
        insurance_rdf, insurance_sdf, column=col, label1="Real", label2="Synthetic"
    )

## Data Preprocessing

In [ ]:
# convert categorical data to categories
insurance_rdf[["SEX", "SMOKER", "REGION"]] = insurance_sdf[["SEX", "SMOKER", "REGION"]].astype(
    "category"
)

In [ ]:
insurance_sdf[["SEX", "SMOKER", "REGION"]] = insurance_sdf[["SEX", "SMOKER", "REGION"]].astype(
    "category"
)

In [ ]:
insurance_sdf1[["SEX", "SMOKER", "REGION"]] = insurance_sdf[["SEX", "SMOKER", "REGION"]].astype(
    "category"
)

In [ ]:
insurance_rdf.dtypes

### One Hot Encoding

In [ ]:
def encode_categorical_features(df):
    """
    One-hot encodes all categorical (object-type) columns in the given DataFrame.

    Parameters:
        df (pd.DataFrame): The input DataFrame.

    Returns:
        pd.DataFrame: A new DataFrame with categorical features one-hot encoded.
    """
    # Select categorical columns
    categorical_columns = df.select_dtypes(include="category").columns

    # Create column transformer for one-hot encoding
    categorical_transformer = ColumnTransformer(
        transformers=[("cat", OneHotEncoder(drop="first"), categorical_columns)],
        remainder="passthrough",
    )

    # Create and apply pipeline
    pipeline = Pipeline(steps=[("preprocess", categorical_transformer)])
    processed_arr = pipeline.fit_transform(df)

    # Get feature names
    ohe = pipeline.named_steps["preprocess"].named_transformers_["cat"]
    encoded_feature_names = ohe.get_feature_names_out(categorical_columns)
    other_feature_names = [col for col in df.columns if col not in categorical_columns]
    all_feature_names = list(encoded_feature_names) + list(other_feature_names)

    return pd.DataFrame(processed_arr, columns=all_feature_names)

In [ ]:
processed_rdf = encode_categorical_features(insurance_rdf)

In [ ]:
processed_rdf.head()

In [ ]:
processed_sdf = encode_categorical_features(insurance_sdf)

In [ ]:
processed_sdf.head()

## Dimensionality Reduction Algorithms

In [ ]:
# def compare_embeddings(
#     real_data,
#     synthetic_data,
#     algorithm,
#     n_real=None,
#     random_state=None,
#     **kwargs,
# ):
#     """
#     Compare real and synthetic data using a specified embedding algorithm and plot the result.

#     Args:
#         real_data: np.ndarray or pd.DataFrame of real samples.
#         synthetic_data: np.ndarray or pd.DataFrame of synthetic samples.
#         algorithm: string, embedding algorithm to run ('tsne' or 'umap').
#         n_real: int or None, total number of samples (real + synthetic).
#         random_state: int or None, random seed.
#         **kwargs: algorithm-specific parameters passed directly to the embedding method.

#     Returns:
#         elapsed time in seconds.
#     """
#     supported_algorithms = {"tsne", "umap"}
#     algorithm = algorithm.lower()
#     if algorithm not in supported_algorithms:
#         raise ValueError(f"Unsupported algorithm: {algorithm}. Supported: {supported_algorithms}")

#     rng = np.random.default_rng(random_state)

#     real_data = real_data.values if isinstance(real_data, pd.DataFrame) else real_data
#     synthetic_data = (
#         synthetic_data.values if isinstance(synthetic_data, pd.DataFrame) else synthetic_data
#     )

#     target_size = n_real if n_real is not None else min(len(real_data), len(synthetic_data))
#     n_half = target_size // 2

#     real_sample_size = min(len(real_data), n_half)
#     real_indices = rng.choice(len(real_data), real_sample_size, replace=False)
#     selected_real = real_data[real_indices]

#     synth_needed = target_size - real_sample_size
#     synth_sample_size = synth_needed
#     synth_indices = rng.choice(len(synthetic_data), synth_sample_size, replace=False)
#     selected_synth = synthetic_data[synth_indices]

#     print(f"Size of data - real: {len(selected_real)}, synthetic: {len(selected_synth)}")
#     combined_data = np.vstack([selected_real, selected_synth])
#     labels = np.array(["Real"] * len(selected_real) + ["Synthetic"] * len(selected_synth))

#     # 2D embedding
#     kwargs["n_components"] = 2
#     if random_state is not None:
#         kwargs.setdefault("random_state", random_state)

#     start = time.time()
#     if algorithm == "tsne":
#         model = TSNE(**kwargs)
#         embedding = model.fit_transform(combined_data)
#     else:  # umap
#         model = UMAP(**kwargs)
#         embedding = model.fit_transform(combined_data)
#     elapsed_time = time.time() - start

#     plt.figure(figsize=(8, 6))
#     for label in ["Real", "Synthetic"]:
#         idx = labels == label
#         plt.scatter(embedding[idx, 0], embedding[idx, 1], label=label, alpha=0.5, s=10)

#     plt.title(
#         f"{algorithm.upper()} (Time={elapsed_time:.2f}s)",
#         weight="bold",
#     )
#     plt.xlabel(f"{algorithm.upper()} 1")
#     plt.ylabel(f"{algorithm.upper()} 2")
#     plt.legend()
#     plt.tight_layout()
#     plt.show()

#     return elapsed_time

In [ ]:
# def compare_embeddings(
#     real_data,
#     synthetic_data,
#     algorithm,
#     n_components=2,
#     n_real=None,
#     random_state=None,
#     **kwargs,
# ):
#     """
#     Compare real and synthetic data using a specified embedding algorithm and produce
#     interactive Plotly scatter plots in 2D or 3D.

#     Args:
#         real_data: np.ndarray or pd.DataFrame of real samples.
#         synthetic_data: np.ndarray or pd.DataFrame of synthetic samples.
#         algorithm: str, 'tsne' or 'umap'.
#         n_components: int, 2 or 3. Number of embedding dimensions/axes.
#         n_real: int or None. Total number of samples (real + synthetic).
#         random_state: int or None. Random seed.
#         **kwargs: passed directly to the TSNE/UMAP constructor
#                   (except n_components/random_state).

#     Returns:
#         elapsed_time: float, seconds taken for fit_transform.
#         fig: Plotly figure (2D or 3D).
#     """
#     algorithm = algorithm.lower()
#     if algorithm not in {"tsne", "umap"}:
#         raise ValueError(f"Unsupported algorithm: {algorithm!r}")

#     # unwrap DataFrames
#     real = real_data.values if isinstance(real_data, pd.DataFrame) else real_data
#     synth = synthetic_data.values if isinstance(synthetic_data, pd.DataFrame) else synthetic_data

#     # sampling
#     rng = np.random.default_rng(random_state)
#     target = n_real or min(len(real), len(synth))
#     half = target // 2
#     real_n = min(len(real), half)
#     synth_n = target - real_n

#     real_idx = rng.choice(len(real), real_n, replace=False)
#     synth_idx = rng.choice(len(synth), synth_n, replace=False)

#     X = np.vstack([real[real_idx], synth[synth_idx]])
#     labels = np.array(["Real"] * real_n + ["Synthetic"] * synth_n)

#     # set up embedding
#     params = dict(kwargs)
#     params["n_components"] = n_components
#     if random_state is not None:
#         params.setdefault("random_state", random_state)

#     # run and time
#     start = time.time()
#     model = TSNE(**params) if algorithm == "tsne" else UMAP(**params)
#     embedding = model.fit_transform(X)
#     elapsed = time.time() - start

#     # put into DataFrame for plotly
#     df_plot = pd.DataFrame(
#         embedding[:, :n_components],
#         columns=[f"{algorithm.upper()}_{i+1}" for i in range(n_components)],
#     )
#     df_plot["label"] = labels

#     title = f"{algorithm.upper()} ({n_components}D) — {elapsed:.2f}s"

#     # interactive plot
#     if n_components == 2:
#         fig = px.scatter(
#             df_plot,
#             x=f"{algorithm.upper()}_1",
#             y=f"{algorithm.upper()}_2",
#             color="label",
#             title=title,
#             labels={"label": "Data Type"},
#         )
#         fig.update_traces(marker=dict(size=5, opacity=0.7))

#     elif n_components == 3:
#         fig = px.scatter_3d(
#             df_plot,
#             x=f"{algorithm.upper()}_1",
#             y=f"{algorithm.upper()}_2",
#             z=f"{algorithm.upper()}_3",
#             color="label",
#             title=title,
#             labels={"label": "Data Type"},
#         )
#         fig.update_traces(marker=dict(size=3, opacity=0.7))

#     else:
#         raise ValueError("n_components must be 2 or 3")

#     fig.show()

#     return elapsed

In [ ]:
def compare_embeddings(
    real_data,
    synthetic_data,
    algorithm,
    n_components=2,
    n_real=None,
    random_state=None,
    **kwargs,
):
    """
    Compare real and synthetic data using a specified embedding algorithm and produce
    interactive Plotly scatter plots in 2D or 3D.

    Args:
        real_data: np.ndarray or pd.DataFrame of real samples.
        synthetic_data: np.ndarray or pd.DataFrame of synthetic samples.
        algorithm: str, 'tsne' or 'umap'.
        n_components: int, 2 or 3. Number of embedding dimensions/axes.
        n_real: int or None. Total number of samples (real + synthetic).
        random_state: int or None. Random seed.
        **kwargs: passed directly to the TSNE/UMAP constructor
                  (except n_components/random_state).

    Returns:
        elapsed_time: float, seconds taken for fit_transform.
        fig: Plotly figure (2D or 3D).
    """
    algorithm = algorithm.lower()
    if algorithm not in {"tsne", "umap"}:
        raise ValueError(f"Unsupported algorithm: {algorithm!r}")

    # unwrap DataFrames
    real = real_data.values if isinstance(real_data, pd.DataFrame) else real_data
    synth = synthetic_data.values if isinstance(synthetic_data, pd.DataFrame) else synthetic_data

    # sampling
    rng = np.random.default_rng(random_state)
    target = n_real or min(len(real), len(synth))
    half = target // 2
    real_n = min(len(real), half)
    synth_n = target - real_n

    real_idx = rng.choice(len(real), real_n, replace=False)
    synth_idx = rng.choice(len(synth), synth_n, replace=False)

    X = np.vstack([real[real_idx], synth[synth_idx]])
    labels = np.array(["Real"] * real_n + ["Synthetic"] * synth_n)

    # set up embedding
    params = dict(kwargs)
    params["n_components"] = n_components
    if random_state is not None:
        params.setdefault("random_state", random_state)

    # run and time
    start = time.time()
    model = TSNE(**params) if algorithm == "tsne" else UMAP(**params)
    embedding = model.fit_transform(X)
    elapsed = time.time() - start

    # put into DataFrame for plotly
    df_plot = pd.DataFrame(
        embedding[:, :n_components],
        columns=[f"{algorithm.upper()}_{i+1}" for i in range(n_components)],
    )
    df_plot["label"] = labels

    title = f"{algorithm.upper()} ({n_components}D) — {elapsed:.2f}s"

    # interactive plot
    if n_components == 2:
        fig = px.scatter(
            df_plot,
            x=f"{algorithm.upper()}_1",
            y=f"{algorithm.upper()}_2",
            color="label",
            title=title,
            labels={"label": "Data Type"},
        )
        fig.update_traces(marker=dict(size=5, opacity=0.7))

    elif n_components == 3:
        fig = px.scatter_3d(
            df_plot,
            x=f"{algorithm.upper()}_1",
            y=f"{algorithm.upper()}_2",
            z=f"{algorithm.upper()}_3",
            color="label",
            title=title,
            labels={"label": "Data Type"},
        )
        fig.update_traces(marker=dict(size=3, opacity=0.7))

    else:
        raise ValueError("n_components must be 2 or 3")

    fig.update_layout(width=800, height=600)

    fig.show()

    return elapsed

In [ ]:
def add_noise(data, noise_std, random_state=None):
    rng = np.random.default_rng(random_state)
    noise = rng.normal(loc=0, scale=noise_std, size=data.shape)
    return data + noise

In [ ]:
# adding noise of  5 SD
noisy_synth = add_noise(processed_sdf.values, noise_std=5, random_state=42)

In [ ]:
# def compare_embeddings_with_noise(
#     real_data,
#     synthetic_data,
#     algorithm,
#     n_components=2,
#     n_real=None,
#     random_state=None,
#     noise_std=0.0,  # 👈 New parameter
#     **kwargs,
# ):
#     """
#     Compare real and synthetic data using a specified embedding algorithm and produce
#     interactive Plotly scatter plots in 2D or 3D, with optional noise added to synthetic data.
#     """
#     algorithm = algorithm.lower()
#     if algorithm not in {"tsne", "umap"}:
#         raise ValueError(f"Unsupported algorithm: {algorithm!r}")

#     # unwrap DataFrames
#     real = real_data.values if isinstance(real_data, pd.DataFrame) else real_data
#     synth = synthetic_data.values if isinstance(synthetic_data, pd.DataFrame) else synthetic_data

#     # --- Add Gaussian noise to synthetic data ---
#     if noise_std > 0:
#         rng = np.random.default_rng(random_state)
#         noise = rng.normal(loc=0, scale=noise_std, size=synth.shape)
#         synth = synth + noise

#     # sampling
#     rng = np.random.default_rng(random_state)
#     target = n_real or min(len(real), len(synth))
#     half = target // 2
#     real_n = min(len(real), half)
#     synth_n = target - real_n

#     real_idx = rng.choice(len(real), real_n, replace=False)
#     synth_idx = rng.choice(len(synth), synth_n, replace=False)

#     X = np.vstack([real[real_idx], synth[synth_idx]])
#     labels = np.array(["Real"] * real_n + ["Synthetic"] * synth_n)

#     # set up embedding
#     params = dict(kwargs)
#     params["n_components"] = n_components
#     if random_state is not None:
#         params.setdefault("random_state", random_state)

#     start = time.time()
#     model = TSNE(**params) if algorithm == "tsne" else UMAP(**params)
#     embedding = model.fit_transform(X)
#     elapsed = time.time() - start

#     # put into DataFrame for plotly
#     df_plot = pd.DataFrame(
#         embedding[:, :n_components],
#         columns=[f"{algorithm.upper()}_{i+1}" for i in range(n_components)],
#     )
#     df_plot["label"] = labels

#     title = f"{algorithm.upper()} ({n_components}D) — {elapsed:.2f}s — Noise STD: {noise_std}"

#     if n_components == 2:
#         fig = px.scatter(
#             df_plot,
#             x=f"{algorithm.upper()}_1",
#             y=f"{algorithm.upper()}_2",
#             color="label",
#             title=title,
#             labels={"label": "Data Type"},
#         )
#         fig.update_traces(marker=dict(size=5, opacity=0.7))

#     elif n_components == 3:
#         fig = px.scatter_3d(
#             df_plot,
#             x=f"{algorithm.upper()}_1",
#             y=f"{algorithm.upper()}_2",
#             z=f"{algorithm.upper()}_3",
#             color="label",
#             title=title,
#             labels={"label": "Data Type"},
#         )
#         fig.update_traces(marker=dict(size=3, opacity=0.7))

#     else:
#         raise ValueError("n_components must be 2 or 3")

#     fig.update_layout(width=800, height=600)
#     fig.show()

#     return elapsed

### Interpretabiltiy

In [ ]:
# runs = performance_df["run"].unique()
# algorithms = performance_df["algorithm"].unique()

# for run in runs:
#     run_df = performance_df[performance_df["run"] == run]

#     fig, axes = plt.subplots(1, len(run_df), figsize=(5 * len(run_df), 4))

#     if len(run_df) == 1:
#         axes = [axes]

#     for ax, (_, row) in zip(axes, run_df.iterrows()):
#         data = row["data"]
#         algorithm = row["algorithm"]

#         sns.scatterplot(x=data[:, 0], y=data[:, 1], s=30, ax=ax)
#         ax.set_title(f"{algorithm} - Run {run}")
#         ax.set_xlabel("Component 1")
#         ax.set_ylabel("Component 2")

#     plt.tight_layout()
#     plt.show()

In [ ]:
# %%timeit -n 1 -r 1

# dataset_sizes = [100, 500, 1000, 2000, 3000, 5000, 10000, 15000, 20000]
# times = []

# for size in dataset_sizes:
#     print(f"Running t-SNE for size {size}")
#     elapsed_time = tsne_compare(processed_rdf, processed_sdf, n_real=size, n_synth=size)
#     times.append(elapsed_time)

In [ ]:
# for size in dataset_sizes:
#     print(f"\nRunning t-SNE {runs_per_size} times for sample size {size}")
#     times[size] = []

#     for run in range(runs_per_size):
#         print(f"  Run {run + 1}...")
#         elapsed_time = tsne_compare(
#             processed_rdf,
#             processed_sdf,
#             n_real=size,
#             perplexity=15,
#             learning_rate=15,
#         )
#         times[size].append(elapsed_time)

In [ ]:
dataset_sizes = [1000, 2000, 5000, 10000, 15000, 20000, 25000, 30000]

In [ ]:
runs_per_size = 5
times = {}

In [ ]:
# for size in dataset_sizes:
#     print(f"\nRunning {runs_per_size} times for sample size {size}")
#     times[size] = []

#     for run in range(runs_per_size):
#         print(f"  Run {run + 1}...")
#         elapsed_time = compare_embeddings_with_noise(
#             processed_rdf,
#             noisy_synth,
#             algorithm="umap",
#             n_components=2,
#             noise_std=2,
#             n_real=size,
#             random_state=run,
#             n_neighbors=15,
#         )
#         times[size].append(elapsed_time)

In [ ]:
for size in dataset_sizes:
    print(f"\nRunning {runs_per_size} times for sample size {size}")
    times[size] = []

    for run in range(runs_per_size):
        print(f"  Run {run + 1}...")
        elapsed_time = compare_embeddings(
            processed_rdf,
            noisy_synth,
            algorithm="umap",
            n_real=size,
            n_neighbors=15,
            random_state=run,
        )
        times[size].append(elapsed_time)

### Performance

In [ ]:
# all_algorithms = [
#     UMAP(),
#     LocallyLinearEmbedding(),
#     SpectralEmbedding(),
#     Isomap(n_neighbors=10),
#     TSNE(),
#     # MDS(),
# ]

In [ ]:
# def benchmark_algorithms(algorithms, data, n_runs=5, verbose=False):
#     """
#     Runs each algorithm multiple times on the data and returns a DataFrame
#     with runtime and output data per run.

#     Parameters:
#         algorithms: list of algorithms
#         data: input data (DataFrame)
#         n_runs: number of times to run each algorithm

#     Returns:
#         performance_df: DataFrame with columns: algorithm, run, time, data
#     """
#     records = []

#     for algorithm in algorithms:
#         alg_name = str(algorithm).split("()")[0]
#         if verbose:
#             print(f"{alg_name} running now")
#         for run in range(n_runs):
#             if verbose:
#                 print(f"current run {run}")
#             start_time = time.time()
#             fit = algorithm.fit_transform(data)
#             elapsed_time = time.time() - start_time
#             records.append(
#                 {"algorithm": alg_name, "run": run + 1, "time": elapsed_time, "data": fit}
#             )

#     performance_df = pd.DataFrame(records)
#     return performance_df

In [ ]:
# start = time.time()
# performance_df = benchmark_algorithms(
#     all_algorithms, processed_insurance_rdf, n_runs=5, verbose=True
# )
# print(f"Time to run benchmark algorithm", time.time() - start)

In [ ]:
# sns.lineplot(data=performance_df, x="run", y="time", hue="algorithm", marker="o")
# plt.title("Runtime per Run for Each Algorithm")
# plt.ylabel("Time (s)")
# plt.xlabel("Run")
# plt.legend(title="Algorithm")
# plt.grid(True)
# plt.show()

In [ ]:
dataset_sizes = np.array(list(times.keys()))
mean_times = np.array([np.mean(times[size]) for size in dataset_sizes])

In [ ]:
log_mean_times = np.log(mean_times)

# Fit linear regression on mean times
X = dataset_sizes.reshape(-1, 1)
y = log_mean_times

model = LinearRegression()
model.fit(X, y)

# Prediction line
x_line = np.linspace(min(dataset_sizes), max(dataset_sizes), 500).reshape(-1, 1)
y_line = model.predict(x_line)

In [ ]:
# Scatter points
scatter = go.Scatter(
    x=dataset_sizes,
    y=log_mean_times,
    mode="markers",
    marker=dict(size=6, opacity=0.6, color="blue"),
    name="Log(Mean Run Time)",
)

# Regression line
line = go.Scatter(
    x=x_line.flatten(),  # x_line was a 2D array
    y=y_line,
    mode="lines",
    line=dict(color="red", width=2),
    name="Linear Fit",
)

# Layout
layout = go.Layout(
    title="Linear Regression on Log-Transformed Mean Run Times",
    xaxis=dict(title="Dataset Size"),
    yaxis=dict(title="Log(Mean Run Time)"),
    width=800,
    height=500,
)

# Combine and plot
fig = go.Figure(data=[scatter, line], layout=layout)
fig.show()

#### Predicted Time to process 100k - 1m rows

In [ ]:
x = np.linspace(100000, 1000000, 10)
y = model.predict(x.reshape(-1, 1))

In [ ]:
# Plot
plt.figure(figsize=(8, 5))
plt.scatter(x, y, marker="o", linestyle="-", color="blue", label="Run Time")
plt.xlabel("Dataset Size")
plt.ylabel("Run Time")
plt.title("Run Time vs Dataset Size")
plt.grid(True)
plt.legend()

plt.gca().xaxis.set_major_formatter(ScalarFormatter(useMathText=True))
plt.ticklabel_format(style="plain", axis="x")

plt.tight_layout()
plt.show()

In [ ]:
# plt.plot(dataset_sizes, times, marker="o")
# plt.title("t-SNE Runtime vs. Dataset Size")
# plt.xlabel("Number of Samples per Dataset (Real + Synthetic)")
# plt.ylabel("Elapsed Time (seconds)")
# plt.tight_layout()
# plt.show()

In [ ]:
# x_vals = []
# y_vals = []

# for size, run_times in times.items():
#     x_vals.extend([size] * len(run_times))
#     y_vals.extend(run_times)

In [ ]:
# x_vals = np.array(x_vals)
# y_vals = np.array(y_vals)

In [ ]:
# # figure
# plt.figure(figsize=(10, 6))

# # scatter of all individual runs
# sns.scatterplot(x=x_vals, y=y_vals, hue=x_vals, palette="viridis", alpha=0.2, s=1, legend=False)

# # line for each of the runs
# unique_sizes = np.unique(x_vals)
# for run_idx in range(runs_per_size):
#     run_times = y_vals[run_idx::runs_per_size]
#     plt.plot(
#         unique_sizes, run_times, marker="o", linewidth=1.5, alpha=0.6, label=f"Run {run_idx+1}"
#     )

# plt.title("U-Map Execution Time vs Dataset Size (Individual Runs Connected)")

# plt.xlabel("Dataset Size")
# plt.ylabel("Execution Time (seconds)")
# plt.grid(True)
# plt.legend(title="Run Number", loc="upper left", bbox_to_anchor=(1.02, 1))
# plt.tight_layout()
# plt.show()

### 3D Plots

In [ ]:
# for size in dataset_sizes:
#     print(f"\nRunning {runs_per_size} times for sample size {size}")
#     times[size] = []

#     for run in range(runs_per_size):
#         print(f"  Run {run + 1}...")
#         elapsed_time = compare_embeddings(
#             processed_rdf,
#             processed_sdf,
#             algorithm="umap",
#             n_real=size,
#             n_neighbors=15,
#             n_components=3,
#             # random_state=run,
#         )
#         times[size].append(elapsed_time)

In [ ]:
# x_vals = []
# y_vals = []

# for size, run_times in times.items():
#     x_vals.extend([size] * len(run_times))
#     y_vals.extend(run_times)

In [ ]:
# x_vals = np.array(x_vals)
# y_vals = np.array(y_vals)

In [ ]:
# # figure
# plt.figure(figsize=(10, 6))

# # scatter of all individual runs
# sns.scatterplot(x=x_vals, y=y_vals, hue=x_vals, palette="viridis", alpha=0.2, s=1, legend=False)

# # line for each of the runs
# unique_sizes = np.unique(x_vals)
# for run_idx in range(runs_per_size):
#     run_times = y_vals[run_idx::runs_per_size]
#     plt.plot(
#         unique_sizes, run_times, marker="o", linewidth=1.5, alpha=0.6, label=f"Run {run_idx+1}"
#     )

# plt.title("U-Map Execution Time vs Dataset Size (Individual Runs Connected)")

# plt.xlabel("Dataset Size")
# plt.ylabel("Execution Time (seconds)")
# plt.grid(True)
# plt.legend(title="Run Number", loc="upper left", bbox_to_anchor=(1.02, 1))
# plt.tight_layout()
# plt.show()

In [ ]:
# dataset_sizes = np.array(list(times.keys()))
# mean_times = np.array([np.mean(times[size]) for size in dataset_sizes])

In [ ]:
# plt.figure(figsize=(10, 6))
# sns.lineplot(
#     x=dataset_sizes,
#     y=mean_times,
#     color="blue",
#     alpha=0.6,
# )

# plt.title("U-Map Mean Execution Time vs Dataset Size")
# plt.xlabel("Dataset Size")
# plt.ylabel("Execution Time (seconds)")
# plt.grid(True)


# plt.tight_layout()
# plt.show()

### GPU



#### Interpretability


In [ ]:
# def tsne_compare_gpu(
#     real_data,
#     synthetic_data,
#     perplexity=15,
#     learning_rate=10,
#     n_real=None,
#     n_synth=None,
#     random_state=42,
# ):
#     """
#     Compare real and synthetic data using t-SNE embedding and plot the result.
#     Automatically balances 50% real and 50% synthetic. If not enough real data,
#     fills the gap with synthetic data.
#     Returns: elapsed time in seconds.
#     """
#     rng = np.random.default_rng(random_state)

#     # Convert to NumPy arrays if needed
#     real_data = real_data.values if isinstance(real_data, pd.DataFrame) else real_data
#     synthetic_data = (
#         synthetic_data.values
#         if isinstance(synthetic_data, pd.DataFrame)
#         else synthetic_data
#     )

#     # Determine target sample size
#     target_size = (
#         n_real if n_real is not None else min(len(real_data), len(synthetic_data))
#     )
#     n_half = target_size // 2

#     # Select real samples (as many as possible up to n_half)
#     real_sample_size = min(len(real_data), n_half)
#     real_indices = rng.choice(len(real_data), real_sample_size, replace=False)
#     selected_real = real_data[real_indices]

#     # Select synthetic samples: at least n_half + any shortfall from real
#     synth_needed = target_size - real_sample_size
#     synth_sample_size = synth_needed
#     synth_indices = rng.choice(len(synthetic_data), synth_sample_size, replace=False)
#     selected_synth = synthetic_data[synth_indices]

#     # Combine data and labels
#     print(
#         f"Size of data - real: {len(selected_real)}, synthethic:{len(selected_synth)}"
#     )
#     combined_data = np.vstack([selected_real, selected_synth])
#     labels = np.array(
#         ["Real"] * len(selected_real) + ["Synthetic"] * len(selected_synth)
#     )

#     # Start timing
#     start = time.time()

#     tsne = TSNE_GPU(n_components=2, perplexity=perplexity, learning_rate=learning_rate)
#     embedding = tsne.fit_transform(combined_data)

#     elapsed_time = time.time() - start

#     # Plot
#     plt.figure(figsize=(8, 6))
#     for label in ["Real", "Synthetic"]:
#         idx = labels == label
#         plt.scatter(embedding[idx, 0], embedding[idx, 1], label=label, alpha=0.5, s=10)

#     plt.title(
#         f"t-SNE Comparison (Perplexity={perplexity}, LR={learning_rate}, Time={elapsed_time:.2f}s)",
#         weight="bold",
#     )
#     plt.xlabel("t-SNE 1")
#     plt.ylabel("t-SNE 2")
#     plt.legend()
#     plt.tight_layout()
#     plt.show()

#     return elapsed_time

In [ ]:
# # %%timeit -n 1 -r 1

# dataset_sizes = [100, 500, 1000, 2000, 3000, 5000, 10000, 15000, 20000]
# times = []

# for size in dataset_sizes:
#     print(f"Running t-SNE for total sample size {size}")
#     elapsed_time = tsne_compare_gpu(
#         processed_rdf,
#         processed_sdf,
#         n_real=size,
#         perplexity=15,
#         learning_rate=15,
#     )
#     times.append(elapsed_time)

#### Performance

In [ ]:
# plt.plot(dataset_sizes, times, marker="o")
# plt.title("t-SNE GPU Enabled Runtime vs. Dataset Size")
# plt.xlabel("Number of Samples per Dataset (Real + Synthetic)")
# plt.ylabel("Elapsed Time (seconds)")
# plt.tight_layout()
# plt.show()